# Sepsis Prediction - Model Training
**Input:** `X_structured.csv`, `y_sepsis.csv`, `X_tfidf_final.npz` (optional)  
**Goal:** Predict sepsis from data collected in the first 24h of hospital admission  
**Validation:** Stratified 5-fold cross-validation  
**Metrics:** AUROC, AUPRC, Sensitivity, Specificity, Brier Score

## 0. Imports

In [ ]:
!pip install xgboost shap -q

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
import warnings
import os
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, MaxAbsScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    roc_curve, precision_recall_curve,
    confusion_matrix, classification_report
)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    print('XGBoost not installed -- pip install xgboost')
    HAS_XGB = False

try:
    import shap
    HAS_SHAP = True
except ImportError:
    print('SHAP not installed -- pip install shap')
    HAS_SHAP = False

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

import pickle

print('Imports OK')

## 1. Load data

In [ ]:
X_struct = pd.read_csv('../data/X_structured.csv')
y = pd.read_csv('../data/y_sepsis.csv').squeeze()
y = y.astype(int)

# Safety check: drop los_days if present (data leakage)
if 'los_days' in X_struct.columns:
    print('los_days found and removed -- data leakage (only known at discharge)')
    X_struct = X_struct.drop(columns=['los_days'])

print(f'X_structured shape : {X_struct.shape}')
print(f'y shape            : {y.shape}')
print(f'Sepsis prevalence  : {y.mean()*100:.1f}%  ({y.sum():,} cases / {len(y):,} total)')
print(f'Missing values     : {X_struct.isnull().sum().sum():,} ({X_struct.isnull().mean().mean()*100:.1f}% of cells)')

bio_cols = [c for c in X_struct.columns if any(c.startswith(p) for p in ['min_','max_','mean_','first_'])]
print(f'Biology features   : {len(bio_cols)}')
if len(bio_cols) == 0:
    print('No biology features -- re-run build_sepsis_dataset.ipynb section 5')

n_pos = int(y.sum())
n_neg = int((y == 0).sum())
assert n_pos > 0, 'No sepsis cases in y -- check the label merge in build notebook'
print(f'Positive cases: {n_pos}')

recommended_splits = min(5, n_pos)
recommended_splits = max(2, recommended_splits)
if n_pos < 10:
    print(f'Few positive cases ({n_pos}) -- n_splits adjusted to {recommended_splits}')
else:
    print(f'Recommended n_splits: {recommended_splits}')

# TF-IDF features (optional)
USE_TFIDF = os.path.exists('../data/X_tfidf_final.npz')
if USE_TFIDF:
    X_tfidf = sp.load_npz('../data/X_tfidf_final.npz')
    print(f'TF-IDF shape: {X_tfidf.shape}')
    if X_tfidf.shape[0] != X_struct.shape[0]:
        print('TF-IDF / X_structured size mismatch -- TF-IDF disabled')
        USE_TFIDF = False
    else:
        print('TF-IDF loaded')
else:
    print('../data/X_tfidf_final.npz not found -- training on structured features only')

## 2. Quick data exploration

In [ ]:
df_explore = X_struct.copy()
df_explore['sepsis'] = y.values

print('=== Median by group (sepsis vs. control) ===')
print(df_explore.groupby('sepsis')[X_struct.columns[:10]].median().T.to_string())

missing_pct = X_struct.isnull().mean().sort_values(ascending=False)
print('\n=== Top 10 features with most missing values ===')
print(missing_pct.head(10).apply(lambda x: f'{x*100:.1f}%'))

n_pos = int(y.sum())
n_neg = int((y == 0).sum())
scale_pos_weight = n_neg / n_pos
print(f'\n=== Class imbalance ===')
print(f'Positive (sepsis) : {n_pos:,}')
print(f'Negative          : {n_neg:,}')
print(f'scale_pos_weight  : {scale_pos_weight:.1f}')

## 3. Feature preparation

Structured features are separated into binary and continuous subsets. When TF-IDF features are available, the structured matrix is first imputed and scaled with `MaxAbsScaler` (sparse-compatible, no centering) before being concatenated with the sparse TF-IDF block.

In [ ]:
binary_cols = [c for c in X_struct.columns if X_struct[c].dropna().isin([0, 1]).all()]
continuous_cols = [c for c in X_struct.columns if c not in binary_cols]

print(f'Continuous features (structured): {len(continuous_cols)}')
print(f'Binary features    (structured): {len(binary_cols)}')

X_np_struct = X_struct.values.astype(np.float32)
y_np = y.values.astype(int)

if USE_TFIDF:
    imputer_pre = SimpleImputer(strategy='median')
    X_struct_imp = imputer_pre.fit_transform(X_np_struct)
    # MaxAbsScaler: sparse-compatible (no centering)
    scaler_pre = MaxAbsScaler()
    X_struct_scaled = scaler_pre.fit_transform(X_struct_imp)

    X_full = sp.hstack([
        sp.csr_matrix(X_struct_scaled),
        X_tfidf
    ])
    print(f'X_full (struct + TF-IDF) shape: {X_full.shape}')
    X_np = X_full
else:
    X_full = None
    X_np = X_np_struct
    print(f'X_np = structured only: {X_np.shape}')

print('Features ready')

## 4. Model definitions

All pipelines use `MaxAbsScaler` rather than `RobustScaler` because `RobustScaler` subtracts the median, which destroys sparsity and is incompatible with sparse TF-IDF matrices. `scale_pos_weight` for XGBoost is set to the negative/positive ratio to handle class imbalance.

In [ ]:
preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  MaxAbsScaler())
])

n_pos = int(y_np.sum())
n_neg = int((y_np == 0).sum())
spw   = round(n_neg / n_pos, 1)

models = {
    'Logistic Regression': Pipeline([
        ('pre', preprocessor),
        ('clf', LogisticRegression(
            class_weight='balanced',
            max_iter=1000,
            solver='saga',  # saga supports sparse matrices
            C=0.1,
            random_state=42,
            n_jobs=-1
        ))
    ]),
    'Random Forest': Pipeline([
        ('pre', preprocessor),
        ('clf', RandomForestClassifier(
            n_estimators=300,
            class_weight='balanced',
            max_depth=8,
            min_samples_leaf=20,
            n_jobs=-1,
            random_state=42
        ))
    ]),
    'Gradient Boosting': Pipeline([
        ('pre', preprocessor),
        ('clf', GradientBoostingClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            random_state=42
        ))
    ]),
}

if HAS_XGB:
    models['XGBoost'] = Pipeline([
        ('pre', preprocessor),
        ('clf', XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=5,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=spw,
            eval_metric='aucpr',
            random_state=42,
            n_jobs=-1
        ))
    ])

print(f'{len(models)} models defined: {list(models.keys())}')
print(f'scale_pos_weight = {spw}')

## 5. Cross-validation

To keep runtimes manageable I subsample to 50,000 patients while preserving class prevalence, and use 2-fold CV for the comparison step. Only Logistic Regression and XGBoost are compared here; the best model is then trained on the full dataset in section 6.

In [ ]:
n_pos = int(y_np.sum())
print(f'Positives={n_pos} | Shape={X_np.shape} | sparse={sp.issparse(X_np)}')

# Subsample to 50k patients for speed (keep all positives)
CV_SAMPLE = 50_000
rng = np.random.RandomState(42)

pos_idx = np.where(y_np == 1)[0]
neg_idx = np.where(y_np == 0)[0]

n_neg_sample = min(CV_SAMPLE - len(pos_idx), len(neg_idx))
neg_sample = rng.choice(neg_idx, size=n_neg_sample, replace=False)
sample_idx = np.concatenate([pos_idx, neg_sample])
rng.shuffle(sample_idx)

X_cv = X_np[sample_idx]
y_cv = y_np[sample_idx]
print(f'CV subsample: {X_cv.shape} | prevalence={y_cv.mean()*100:.1f}%')

cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)
scoring = ['roc_auc', 'average_precision']

# Compare the two fastest models
models_cv = {k: v for k, v in models.items() if k in ['Logistic Regression', 'XGBoost']}

results = {}
for name, pipe in models_cv.items():
    print(f'Training {name}...', end=' ', flush=True)
    try:
        scores = cross_validate(
            pipe, X_cv, y_cv,
            cv=cv,
            scoring=scoring,
            n_jobs=1,
            return_train_score=False,
            error_score='raise'
        )
        results[name] = {
            'AUROC': scores['test_roc_auc'],
            'AUPRC': scores['test_average_precision'],
        }
        print(f"AUROC={scores['test_roc_auc'].mean():.3f} | AUPRC={scores['test_average_precision'].mean():.3f}")
    except Exception as e:
        print(f'Failed for {name}: {e}')
        results[name] = {'AUROC': np.array([np.nan]), 'AUPRC': np.array([np.nan])}

print('\n=== Cross-validation results ===')
for name, res in results.items():
    print(f"{name:25s} AUROC={res['AUROC'].mean():.3f} | AUPRC={res['AUPRC'].mean():.3f}")

## 6. Train best model on train/test split

The best model is selected by AUPRC (more informative than AUROC on imbalanced data). Two decision thresholds are computed: an F1-optimal threshold and a clinical threshold that guarantees sensitivity >= 75%.

In [ ]:
valid_results = {n: r for n, r in results.items() if not np.isnan(r['AUPRC'].mean())}
assert len(valid_results) > 0, 'No model trained successfully -- check data'
best_name = max(valid_results, key=lambda n: valid_results[n]['AUPRC'].mean())
print(f'Best model: {best_name}')

best_pipe = models[best_name]

# Stratified train/test split
n_pos = int(y_np.sum())
test_size = max(0.2, 2 / len(y_np))
test_size = min(test_size, 0.3)

X_train, X_test, y_train, y_test = train_test_split(
    X_np, y_np,
    test_size=test_size,
    stratify=y_np,
    random_state=42
)

print(f'Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}')
print(f'Train prevalence: {y_train.mean()*100:.1f}% ({y_train.sum()} positives)')
print(f'Test  prevalence: {y_test.mean()*100:.1f}% ({y_test.sum()} positives)')

assert y_train.sum() > 0, 'No positives in train set'
assert y_test.sum()  > 0, 'No positives in test set'

best_pipe.fit(X_train, y_train)
y_prob = best_pipe.predict_proba(X_test)[:, 1]

# F1-optimal threshold
prec, rec, thresholds_pr = precision_recall_curve(y_test, y_prob)
f1_scores = 2 * prec * rec / (prec + rec + 1e-8)
best_thresh_f1 = float(thresholds_pr[np.argmax(f1_scores[:-1])])

# Clinical threshold: sensitivity >= 75%
TARGET_SENSITIVITY = 0.75
fpr_arr, tpr_arr, roc_thresholds = roc_curve(y_test, y_prob)
eligible_idx = np.where(tpr_arr >= TARGET_SENSITIVITY)[0]
if len(eligible_idx) > 0:
    best_thresh_clinical = float(roc_thresholds[eligible_idx[0]])
    print(f'F1-optimal threshold : {best_thresh_f1:.3f}')
    print(f'Clinical threshold   : {best_thresh_clinical:.3f}  (sensitivity >= {TARGET_SENSITIVITY*100:.0f}%)')
else:
    best_thresh_clinical = best_thresh_f1
    print(f'Target sensitivity unreachable -- using F1 threshold: {best_thresh_f1:.3f}')

best_thresh = best_thresh_clinical

# Test set metrics
auroc = roc_auc_score(y_test, y_prob)
auprc = average_precision_score(y_test, y_prob)
brier = brier_score_loss(y_test, y_prob)

y_pred = (y_prob >= best_thresh).astype(int)
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
npv = tn / (tn + fn) if (tn + fn) > 0 else 0

print(f'\n=== Test set metrics ===')
print(f'AUROC       : {auroc:.3f}')
print(f'AUPRC       : {auprc:.3f}')
print(f'Brier Score : {brier:.3f}  (lower is better, 0 = perfect)')
print(f'Threshold   : {best_thresh:.3f}  (clinical, sensitivity >= {TARGET_SENSITIVITY*100:.0f}%)')
print(f'Sensitivity : {sensitivity:.3f}')
print(f'Specificity : {specificity:.3f}')
print(f'PPV         : {ppv:.3f}')
print(f'NPV         : {npv:.3f}')
print(f'\nConfusion matrix (threshold {best_thresh:.2f}):')
print(f'               Pred 0     Pred 1')
print(f'  True 0 (-)   {tn:>8,}   {fp:>8,}  (TN / FP)')
print(f'  True 1 (+)   {fn:>8,}   {tp:>8,}  (FN / TP)')

## 7. ROC and Precision-Recall curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[0].plot(fpr, tpr, lw=2, label=f'{best_name} (AUROC = {auroc:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
axes[0].set_xlabel('1 - Specificity (FPR)')
axes[0].set_ylabel('Sensitivity (TPR)')
axes[0].set_title('ROC Curve')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Precision-Recall curve
axes[1].plot(rec, prec, lw=2, label=f'{best_name} (AUPRC = {auprc:.3f})')
baseline = y_test.mean()
axes[1].axhline(baseline, color='k', linestyle='--', lw=1, label=f'Baseline ({baseline:.2f})')
axes[1].set_xlabel('Recall (Sensitivity)')
axes[1].set_ylabel('Precision (PPV)')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('roc_pr_curves.png', bbox_inches='tight')
plt.show()
print('Curves saved: roc_pr_curves.png')

## 8. Calibration

The reliability diagram shows how well the predicted probabilities align with observed frequencies. If calibration is poor, Platt scaling (sigmoid) is applied.

In [ ]:
fraction_pos, mean_pred = calibration_curve(y_test, y_prob, n_bins=10)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(mean_pred, fraction_pos, 's-', label=f'{best_name} (Brier={brier:.3f})')
ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Fraction of positives')
ax.set_title('Reliability diagram (Calibration)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('calibration_plot.png', bbox_inches='tight')
plt.show()

# Apply Platt scaling if needed
calibrated_pipe = CalibratedClassifierCV(best_pipe, method='sigmoid', cv='prefit')
calibrated_pipe.fit(X_test, y_test)
y_prob_cal = calibrated_pipe.predict_proba(X_test)[:, 1]
brier_cal = brier_score_loss(y_test, y_prob_cal)
print(f'Brier score before calibration: {brier:.3f}')
print(f'Brier score after  calibration: {brier_cal:.3f}')

## 9. Interpretability with SHAP

SHAP values are computed on a random subsample of 500 test patients to keep runtime short. The summary plot shows the 20 most influential features.

In [ ]:
try:
    import shap

    struct_feature_names = list(X_struct.columns)
    tfidf_feature_names  = [f'tfidf_svd_{i}' for i in range(X_tfidf.shape[1])] if USE_TFIDF else []
    feature_names = struct_feature_names + tfidf_feature_names

    print(f'Structured features : {len(struct_feature_names)}')
    print(f'TF-IDF SVD features : {len(tfidf_feature_names)}')
    print(f'Total feature_names : {len(feature_names)}')

    X_test_pre = best_pipe[:-1].transform(X_test)
    if sp.issparse(X_test_pre):
        X_test_pre = X_test_pre.toarray()

    SHAP_SAMPLE = min(500, X_test_pre.shape[0])
    rng_shap = np.random.RandomState(42)
    shap_idx = rng_shap.choice(X_test_pre.shape[0], size=SHAP_SAMPLE, replace=False)
    X_shap = X_test_pre[shap_idx]

    print(f'SHAP on {SHAP_SAMPLE} examples x {X_shap.shape[1]} features...')
    assert len(feature_names) == X_shap.shape[1], \
        f'Mismatch: {len(feature_names)} names vs {X_shap.shape[1]} features'

    clf = best_pipe.named_steps['clf']
    explainer   = shap.TreeExplainer(clf)
    shap_values = explainer.shap_values(X_shap)

    if isinstance(shap_values, list):
        sv = shap_values[1]
    elif hasattr(shap_values, 'values'):
        sv = shap_values.values
        if sv.ndim == 3:
            sv = sv[:, :, 1]
    else:
        sv = shap_values

    plt.figure(figsize=(10, 8))
    shap.summary_plot(
        sv, X_shap,
        feature_names=feature_names,
        max_display=20,
        show=False
    )
    plt.tight_layout()
    plt.savefig('shap_summary.png', bbox_inches='tight')
    plt.show()
    print('SHAP summary plot saved')

except Exception as e:
    print(f'SHAP failed: {e}')
    # Fallback: tree feature importances
    clf = best_pipe.named_steps['clf']
    if hasattr(clf, 'feature_importances_'):
        struct_feature_names = list(X_struct.columns)
        tfidf_feature_names  = [f'tfidf_svd_{i}' for i in range(X_tfidf.shape[1])] if USE_TFIDF else []
        feature_names = struct_feature_names + tfidf_feature_names
        imp = pd.Series(clf.feature_importances_, index=feature_names).sort_values(ascending=False)
        print('\nTop 20 feature importances:')
        print(imp.head(20).to_string())

## 10. Threshold analysis

Sensitivity, specificity, and PPV across the full range of thresholds. The two vertical lines mark the F1-optimal and clinical thresholds from section 6.

In [ ]:
thresholds_range = np.linspace(0.05, 0.95, 50)
sens_list, spec_list, ppv_list = [], [], []

for t in thresholds_range:
    yp = (y_prob >= t).astype(int)
    tn_, fp_, fn_, tp_ = confusion_matrix(y_test, yp, labels=[0,1]).ravel()
    sens_list.append(tp_ / (tp_ + fn_ + 1e-8))
    spec_list.append(tn_ / (tn_ + fp_ + 1e-8))
    ppv_list.append(tp_ / (tp_ + fp_ + 1e-8))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(thresholds_range, sens_list, label='Sensitivity (recall)', lw=2, color='steelblue')
ax.plot(thresholds_range, spec_list, label='Specificity', lw=2, color='darkorange')
ax.plot(thresholds_range, ppv_list,  label='PPV (precision)', lw=2, color='green')
ax.axvline(best_thresh_f1, color='gray', linestyle=':', lw=1.5,
           label=f'F1-optimal threshold = {best_thresh_f1:.2f}')
ax.axvline(best_thresh_clinical, color='red', linestyle='--', lw=2,
           label=f'Clinical threshold = {best_thresh_clinical:.2f} (sensitivity >= {TARGET_SENSITIVITY*100:.0f}%)')
ax.axhline(TARGET_SENSITIVITY, color='red', linestyle=':', lw=1, alpha=0.5)
ax.set_xlabel('Decision threshold')
ax.set_ylabel('Score')
ax.set_title('Sensitivity / Specificity / PPV by threshold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('threshold_analysis.png', bbox_inches='tight')
plt.show()

print(f'Clinical threshold : {best_thresh_clinical:.3f} (target sensitivity >= {TARGET_SENSITIVITY*100:.0f}%)')
print(f'F1-optimal         : {best_thresh_f1:.3f}')
print('To adjust: change TARGET_SENSITIVITY in section 6')

## 11. Save model

In [ ]:
# Retrain on the full dataset before saving
print('Final training on 100% of data...')
best_pipe.fit(X_np, y_np)

model_path = f'sepsis_model_{best_name.lower().replace(" ", "_")}.pkl'
with open(model_path, 'wb') as f:
    pickle.dump({
        'pipeline':      best_pipe,
        'model_name':    best_name,
        'threshold':     best_thresh,
        'feature_names': X_struct.columns.tolist(),
        'metrics': {
            'auroc':       auroc,
            'auprc':       auprc,
            'brier':       brier,
            'sensitivity': sensitivity,
            'specificity': specificity,
        }
    }, f)

print(f'Model saved: {model_path}')
print('\n=== Final summary ===')
print(f'Model       : {best_name}')
print(f'AUROC       : {auroc:.3f}')
print(f'AUPRC       : {auprc:.3f}')
print(f'Brier Score : {brier:.3f}')
print(f'Sensitivity : {sensitivity:.3f}')
print(f'Specificity : {specificity:.3f}')
print(f'Threshold   : {best_thresh:.3f}')
print('\nFiles produced:')
print(f'  {model_path}')
print('  roc_pr_curves.png')
print('  calibration_plot.png')
print('  threshold_analysis.png')
if HAS_SHAP:
    print('  shap_summary.png')

## 12. Inference on new data

Example showing how to reload the saved model and apply it to a new patient cohort. The input DataFrame must have the same columns as `X_structured.csv`.

In [ ]:
with open(model_path, 'rb') as f:
    saved = pickle.load(f)

pipeline_loaded      = saved['pipeline']
threshold_loaded     = saved['threshold']
feature_names_loaded = saved['feature_names']

# To predict on new data:
# X_new = pd.read_csv('new_patients.csv')
# X_new = X_new[feature_names_loaded]  # ensure column order matches
# proba = pipeline_loaded.predict_proba(X_new.values)[:, 1]
# pred  = (proba >= threshold_loaded).astype(int)

# Quick sanity check on the test set
proba_check = pipeline_loaded.predict_proba(X_test)[:, 1]
pred_check  = (proba_check >= threshold_loaded).astype(int)

print('Model reloaded successfully')
print(f'Predicted sepsis on X_test: {pred_check.sum()} / {len(pred_check)} patients')
print(f'\nExample usage on new data:')
print(f'  X_new = pd.read_csv("new_patients.csv")[feature_names_loaded]')
print(f'  proba = pipeline_loaded.predict_proba(X_new.values)[:, 1]')
print(f'  pred  = (proba >= {threshold_loaded:.3f}).astype(int)')